# ML Laboratuvarı · Ünite 9: Colab + Gemini atölyesi

**Prof. Dr. Mehmet Songur** · ML Laboratuvarı, Modül 0

Bu atölyede gerçek veriyle (ABD TÜFE, FRED: `CPIAUCNS`) çalışacak, kodu **Gemini'ye yazdıracak** ve **denetleyeceksin**.

**Kurallar**
1. **Kurulum** ve **Kontrol** hücrelerini değiştirme; yalnızca çalıştır.
2. Görev hücrelerinde kodu Gemini ile üret (hücredeki Gemini düğmesi ya da Gemini paneli; Colab'ın arayüzüne göre adı değişebilir).
3. Gemini bir kod önerdiğinde önce **Kabul et** (Accept) ile hücreye al; kod henüz çalışmaz. Sonra **çalıştırmadan önce oku**, Ünite 6'daki hata kataloğuyla denetle, gerekirse düzelt ve ancak ondan sonra çalıştır. İstediğin kontroller kodda gerçekten var mı, yoksa Gemini yalnızca sayıları mı yazdırdı?
4. Kontrol hücresi geçerse bir **tamamlama kodu** verir; onu ML Laboratuvarı'ndaki ilgili kutuya yapıştır.

> Kendi kopyanı kaydetmek için: *Dosya → Drive'a kopya kaydet*.

In [ ]:
#@title Kurulum — çalıştır, değiştirme

import io, math, base64, urllib.request
import pandas as pd

def fred_indir(seri_kodu):
    """FRED'den bir seriyi indirir; sütunlar: tarih, <seri_kodu>. FRED eksik değerleri '.' ile gösterir;
    bunlar bilerek NaN yapılır ve sayısı yazdırılır."""
    url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=" + seri_kodu
    with urllib.request.urlopen(url) as yanit:
        df = pd.read_csv(io.BytesIO(yanit.read()))
    df.columns = ["tarih", seri_kodu]
    df["tarih"] = pd.to_datetime(df["tarih"])
    df[seri_kodu] = pd.to_numeric(df[seri_kodu], errors="coerce")
    print(f"{seri_kodu}: {len(df)} satır, {df[seri_kodu].isna().sum()} eksik değer, {df['tarih'].min():%Y-%m} – {df['tarih'].max():%Y-%m}")
    return df

ham = fred_indir("CPIAUCNS").rename(columns={"CPIAUCNS": "endeks"})

def _k(s):
    return base64.b64decode(s).decode()

def _dogru_yillik():
    d = ham.sort_values("tarih").set_index("tarih")["endeks"]
    y = d.pct_change(periods=12, fill_method=None) * 100
    temiz = d.isna().astype(int).rolling(13, min_periods=1).max() == 0   # eksik değer içeren pencereler karşılaştırılmaz
    return y[temiz]

def kontrol_A(df):
    if "yillik" not in df.columns:
        print("❌ 'yillik' adlı bir sütun bulunamadı."); return
    if len(df) != len(ham):
        print(f"❌ Satır sayısı değişmiş: {len(ham)} olmalıydı, {len(df)} oldu. Hangi satırlar silindi ya da çoğaldı?"); return
    if df["tarih"].duplicated().any():
        print("❌ Aynı tarih birden çok kez geçiyor. Satırlar çoğalmış olabilir."); return
    s = df.set_index("tarih")["yillik"].sort_index()
    if not s.iloc[:12].isna().all():
        print("❌ Tarihe göre sıralı tabloda ilk 12 ayın yıllık değeri boş olmalı (12 ay öncesi yok). Hesaptan önce tarihe göre sıraladın mı? Boşluklar doldurulmuş olabilir mi?"); return
    dogru = _dogru_yillik().dropna()
    ortak = s.reindex(dogru.index)
    if ortak.isna().any():
        print(f"❌ Hesaplanabilir {ortak.isna().sum()} ayda 'yillik' boş. Beklenmedik eksik değer var."); return
    if ((ortak * 100 - dogru).abs() < 1e-6).mean() > 0.95:
        print("❌ Değerler oran olarak hesaplanmış. Yüzdeye çevirdin mi?"); return
    fark = (ortak - dogru).abs()
    if fark.max() > 1e-6:
        print(f"❌ {(fark > 1e-6).sum()} ayda değer beklenenden farklı. Hesaptan önce tarihe göre sıraladın mı? periods=12 kullandın mı?"); return
    print("✅ Tüm kontroller geçti. Tamamlama kodun: " + _k("TUwwLUVORi03UUsy"))

def kontrol_B(zirve_ay, zirve_deger):
    d = _dogru_yillik()
    d = d[d.index >= "2000-01-01"].dropna()
    try:
        ay = pd.Timestamp(zirve_ay)
    except Exception:
        print("❌ 'zirve_ay' bir tarihe çevrilemiyor."); return
    try:
        deger = float(zirve_deger)
    except Exception:
        print("❌ 'zirve_deger' bir sayı değil."); return
    if not math.isfinite(deger):
        print("❌ 'zirve_deger' sonlu bir sayı değil (NaN ya da sonsuz). Eksik değerler nasıl ele alındı?"); return
    if ay.to_period("M") != d.idxmax().to_period("M"):
        print("❌ Zirve ay beklenenden farklı. 2000 öncesini çıkardın mı? Yıllık seriyi mi kullandın?"); return
    if abs(deger - d.max()) > 0.05:
        print("❌ Zirve değeri beklenenden farklı. Birim yüzde mi?"); return
    print("✅ Tüm kontroller geçti. Tamamlama kodun: " + _k("TUwwLVpSVi00TVRY"))

# Not: satırlar indirildikleri sırada değil. Denetimde bunu hatırla.
tufe = ham.sample(frac=1, random_state=7).reset_index(drop=True)
print("tufe tablosu:", tufe.shape)
tufe.head()

## Görev A — Yıllık enflasyon

`tufe` tablosuna (sütunlar: `tarih`, `endeks`; aylık) **yıllık enflasyonu yüzde olarak** `yillik` adlı yeni bir sütun olarak ekle.

**Gemini'ye örnek istek (kendi isteğini yazman daha iyi):**
> tufe adlı pandas tablosunda 'tarih' (aylık, datetime) ve 'endeks' sütunları var. Tabloyu tarihe göre sırala; her ay için 12 ay önceye göre yüzde değişimi hesaplayıp 'yillik' adlı yeni sütuna yaz. Başka veri indirme veya uydurma. Satır sayısını ve eksik değer sayısını yazdır.

**Denetim soruları:** Tablo sıralandı mı? Sonuç yüzde mi? `periods=12` mi kullanıldı? Satır kaybı var mı? İlk 12 ay boş mu, sonrasında beklenmedik boşluk var mı?

**Kontrol A neyi sınar?** Satır sayısının değişmediğini; sıralanmış tabloda ilk 12 ayın boş olduğunu; hesaplanabilir her ayda değerin dolu olduğunu ve doğru hesapla **birebir** eşleştiğini; değerlerin oran değil yüzde olduğunu. (FRED'de bir ay eksikse, o ayı içeren 12 aylık pencereler karşılaştırmaya alınmaz.)

In [ ]:
# Görev A: Gemini ile kodu burada üret, oku, sonra çalıştır.

### Kontrol A — çalıştır, değiştirme

In [ ]:
kontrol_A(tufe)

## Görev B — Zirve ay

2000-01 ve sonrasında yıllık enflasyonun **en yüksek** olduğu ayı `zirve_ay` (tarih), değerini `zirve_deger` (yüzde) adlı değişkenlere ata.

**Denetim soruları:** 2000 öncesi dışarıda mı? Eksik değerler (ilk 12 ay) sorun yaratıyor mu? Değer yüzde mi ve sonlu bir sayı mı (NaN değil)?

**Kontrol B neyi sınar?** `zirve_ay`'ın bir tarih ve doğru ay olduğunu; `zirve_deger`'in sonlu bir sayı olduğunu ve doğru değerle eşleştiğini.

In [ ]:
# Görev B: Gemini ile kodu burada üret, oku, sonra çalıştır.

### Kontrol B — çalıştır, değiştirme

In [ ]:
kontrol_B(zirve_ay, zirve_deger)

## Denetim notların

ML Laboratuvarı'ndaki **denetim defterine** geçmeden önce burada kısaca not al:
- Gemini'ye ne istedin?
- Kodunda neyi kontrol ettin?
- Neyi düzelttin, ya da neden değiştirmedin?